# 14b. Memoire paginee d'agent -- fenetre glissante, resume progressif, eviction

**Navigation** : [Version Python](14b_Paginated_Memory-Python.ipynb) | [Version C#](14b_Paginated_Memory-CSharp.ipynb) | [Index](README.md) | [<< Precedent](14_Persistent_Memory.ipynb) | [Suivant >>](15_Tree_of_Thoughts_Search.ipynb)

Ce notebook montre comment borner le contexte rendu d’un agent en combinant fenêtre glissante, troncature cumulative et éviction, tout en gardant les informations anciennes accessibles à la demande.

**Objectifs** : comparer fenêtre glissante, résumé tronqué, éviction FIFO/LRU/importance et pagination ; suivre les identités et les coûts modélisés ; distinguer contexte rendu et stockage complet.

**Prérequis** : listes, files, dictionnaires et complexité algorithmique de base.

**Durée estimée** : 70 minutes.


## Le probleme de la memoire non bornable

Un agent conversationnel qui appelle un LLM sur chaque tour accumule des messages dans la fenetre de contexte du modele. Trois symptomes apparaissent des que la conversation dure :

1. **Cout** : chaque tour peut repayer le contexte déjà transmis, ce qui pèse sur le coût réel d’un service. Les mesures synthétiques plus loin utilisent toutefois des unités de caractères, pas des tokens facturés.
2. **Latence** : la latence d'un appel LLM depend du nombre de tokens d'entree. La conversation ralentit mecaniquement au fil de l'eau.
3. **Degradation de la qualite** : les modeles oublient le milieu de leur contexte (lost-in-the-middle), et les anciens details ne servent plus rien une fois la discussion passee.

La memoire **persistante** de `14_Persistent_Memory.ipynb` (vecteur + similarite cosinus) repond au probleme du **transfer cross-session** : les lecons survivent aux runs et sont retrocedees par similarite. Elle ne repond **pas** au probleme du **cout intra-session** : dans une seule conversation longue, l'historique continue de croitre jusqu'a exploser la fenetre.

Ce notebook introduit la memoire **paginee** : trois mecanismes qui cooperent pour borner la taille de la fenetre de contexte vue par le LLM a chaque tour, tout en preservant les informations anciennes sous une forme accessible a la demande.

## Plan

1. Definitions et vocabulaire : fenetre glissante, resume progressif, eviction, paging.
2. Fenetre glissante (rolling window) : ne garder que les N derniers echanges intacts.
3. Resume progressif : condenser les messages evicques en un resume cumulatif.
4. Politique d'eviction : FIFO, LRU, importance-scored.
5. Memory paging (style Generative Agents / MemGPT) : hierarchie core/archival + retrieval a la demande.
6. Mesure : comparer un cout modélisé en caractères et le nombre de messages retenus sur une longue conversation synthetique, avec et sans pagination.
7. Trois exercices : fenetre parametrique, eviction LRU sur la memoire vectorielle de NB-14, transfer cross-paging multi-domaines.


## 0. Setup -- structures de donnees pures, sans dependance API

Ce notebook est **CPU-pur** : il ne depend d'aucun appel LLM pour fonctionner. La pagination est un mecanisme structurel, et les resumes sont simules par troncature + concatenation (un vrai systeme brancherait ici un appel `chat(...)` de NB-14 ou NB-09 pour generer le resume ; ce notebook montre le **squelette**, le slot d'appel est marque d'un `# HOOK`). Ce portage Rust s'execute sur le kernel evcxr avec la seule bibliotheque standard ; chaque cellule de code redefinit les structures dont elle a besoin.

In [2]:
// Imports du jumeau Python : __future__, collections (deque, OrderedDict),
// dataclasses, typing, json, time. Le portage Rust n'utilise que la
// bibliotheque standard ; les `use` utiles vivent dans les cellules qui s'en
// servent. Le message ci-dessous est recopie tel quel (temoin de comparaison).
println!("Imports OK : dataclasses, collections, json (memoire paginee pure Python, sans API).");

Imports OK : dataclasses, collections, json (memoire paginee pure Python, sans API).


## 1. Definitions et vocabulaire

On formalise les **trois mecanismes** de pagination avant de les implementer. Le vocabulaire suit la litterature Generative Agents (Park et al., 2023) et MemGPT (Packer et al., 2023).

| Mecanisme | Question qu'il resout | Granularite |
|---|---|---|
| **Fenetre glissante** (rolling window) | Combien de messages recents garder intacts ? | Message (1 tour = 1 echange user/assistant) |
| **Resume progressif** (progressive summarization) | Que faire des messages evicques ? | Bloc resume cumulatif |
| **Eviction** (eviction policy) | Lequel supprimer en priorite quand la memoire est pleine ? | FIFO / LRU / importance-scored |
| **Memory paging** (core/archival retrieval) | Comment retrouver une information ancienne sans tout recharger ? | Page (sous-unite de memoire adressable) |

Un agent **reel** combine les quatre. Ici les quatre mécanismes sont présentés séparément. La classe `PagedMemory` illustre seulement les pages et leur sélection ; elle ne les assemble pas en un agent intégré.

## 2. Fenetre glissante (rolling window)

Le mecanisme le plus simple : la memoire courante est un tampon borne de N entrees (en Rust, une `VecDeque` dont l'eviction de tete est explicite) qui conserve les **N derniers messages**. Au-dela, les plus anciens sont automatiquement evicques (FIFO).

**Avantage** : O(1) en insert et en eviction, deterministe.
**Limite** : aucune information n'est preservee au-dela de la fenetre (oubli total). Un message important du tour 3 est perdu au tour N+3.

In [3]:
use std::collections::VecDeque;

// Equivalent du dataclass Python ; ts et importance servent aux politiques d'eviction.
#[derive(Clone, Debug)]
#[allow(dead_code)]
struct Message {
    role: String,   // user / assistant / system / tool
    content: String,
    ts: f64,
    importance: f64,
}

impl Message {
    fn next_ts() -> f64 {
        // Horloge monotone de substitution au time.time() Python (l'ordre seul compte).
        static NEXT_TS: std::sync::atomic::AtomicU64 = std::sync::atomic::AtomicU64::new(0);
        NEXT_TS.fetch_add(1, std::sync::atomic::Ordering::SeqCst) as f64
    }

    fn new(role: &str, content: &str) -> Self {
        Message { role: role.to_string(), content: content.to_string(), ts: Self::next_ts(), importance: 1.0 }
    }
}

struct RollingWindow {
    max: usize,
    buf: VecDeque<Message>,
    evicted_total: usize,
}

impl RollingWindow {
    fn new(max_messages: usize) -> Result<Self, String> {
        if max_messages < 1 {
            return Err("max_messages >= 1 requis.".to_string());
        }
        Ok(RollingWindow { max: max_messages, buf: VecDeque::new(), evicted_total: 0 })
    }

    fn append(&mut self, msg: Message) -> Vec<Message> {
        // Equivalent du deque(maxlen=N) : la tete sort AVANT l'ajout.
        let mut ev = Vec::new();
        if self.buf.len() == self.max {
            ev.push(self.buf.pop_front().unwrap());
        }
        self.buf.push_back(msg);
        self.evicted_total += ev.len();
        ev
    }

    fn len(&self) -> usize {
        self.buf.len()
    }

    fn render(&self) -> String {
        self.buf.iter()
            .map(|m| format!("[{}] {}", m.role, m.content))
            .collect::<Vec<_>>()
            .join("\n")
    }
}

// Repr de liste Python : ['a', 'b'] (apostrophes, virgule + espace).
fn py_list(items: &[&str]) -> String {
    format!("[{}]", items.iter().map(|s| format!("'{}'", s)).collect::<Vec<_>>().join(", "))
}

// Demonstration : 6 messages dans une fenetre de taille 4.
let mut rw = RollingWindow::new(4).unwrap();
for i in 0..6u32 {
    let m = Message::new(if i % 2 == 0 { "user" } else { "assistant" }, &format!("message #{}", i));
    let ev = rw.append(m);
    let label = if ev.is_empty() {
        String::new()
    } else {
        let items: Vec<&str> = ev.iter().map(|e| e.content.as_str()).collect();
        format!("  -> evince(s) : {}", py_list(&items))
    };
    println!("append #{}: taille={}{}", i, rw.len(), label);
}
println!();
println!("Contenu final de la fenetre :");
println!("{}", rw.render());
println!("Total evicques sur la duree de vie : {}", rw.evicted_total);

append #0: taille=1


append #1: taille=2


append #2: taille=3


append #3: taille=4


append #4: taille=4  -> evince(s) : ['message #0']


append #5: taille=4  -> evince(s) : ['message #1']


... (2 outputs left out)

[assistant] message #3


[user] message #4


[assistant] message #5


Total evicques sur la duree de vie : 2


**Lecture de la sortie.** Les messages #0 et #1 ont ete evicques (sortie `evince(s) : ['message #0']` puis `['message #1']` apparaissent sur les appends #4 et #5). Au final, la fenetre contient les messages #2 a #5, dans l'ordre. Le total evicque cumule est 2, et la taille de la fenetre reste bornee a 4 quel que soit le nombre d'appels.

## 3. Resume progressif

Quand un message est evince de la fenetre glissante, son contenu est **perdu**. Pour preserver l'information ancienne sous une forme compacte, on accumule un **resume cumulatif** : a chaque eviction, on concatene le message evince au resume en cours (ou on appelle un LLM pour le paraphraser).

**Strategie simplifiee** (notebook CPU-pur) : troncature a 80 caracteres par message evince, concatenation par saut de ligne. En production, remplacer par `summarize(msg) -> str` (cf `# HOOK`). Seul le rendu peut etre borne par construction ; ici les chunks restent dans une liste et le stockage croît en O(N). Une vraie consolidation periodique serait necessaire pour borner aussi le stockage.

In [4]:
use std::collections::VecDeque;

#[derive(Clone, Debug)]
#[allow(dead_code)]
struct Message {
    role: String,   // user / assistant / system / tool
    content: String,
    ts: f64,
    importance: f64,
}

impl Message {
    fn next_ts() -> f64 {
        // Horloge monotone de substitution au time.time() Python (l'ordre seul compte).
        static NEXT_TS: std::sync::atomic::AtomicU64 = std::sync::atomic::AtomicU64::new(0);
        NEXT_TS.fetch_add(1, std::sync::atomic::Ordering::SeqCst) as f64
    }

    fn new(role: &str, content: &str) -> Self {
        Message { role: role.to_string(), content: content.to_string(), ts: Self::next_ts(), importance: 1.0 }
    }
}

struct RollingWindow {
    max: usize,
    buf: VecDeque<Message>,
    evicted_total: usize,
}

impl RollingWindow {
    fn new(max_messages: usize) -> Result<Self, String> {
        if max_messages < 1 {
            return Err("max_messages >= 1 requis.".to_string());
        }
        Ok(RollingWindow { max: max_messages, buf: VecDeque::new(), evicted_total: 0 })
    }

    fn append(&mut self, msg: Message) -> Vec<Message> {
        let mut ev = Vec::new();
        if self.buf.len() == self.max {
            ev.push(self.buf.pop_front().unwrap());
        }
        self.buf.push_back(msg);
        self.evicted_total += ev.len();
        ev
    }

    fn len(&self) -> usize {
        self.buf.len()
    }

    fn render(&self) -> String {
        self.buf.iter()
            .map(|m| format!("[{}] {}", m.role, m.content))
            .collect::<Vec<_>>()
            .join("\n")
    }
}

struct ProgressiveSummary {
    chunks: Vec<String>,
    max_chars_per_chunk: usize,
}

impl ProgressiveSummary {
    fn new() -> Self {
        ProgressiveSummary { chunks: Vec::new(), max_chars_per_chunk: 80 }
    }

    fn ajouter(&mut self, msg: &Message) {
        // HOOK production : remplacer par summarize(msg.content) (appel LLM).
        let mut c = msg.content.clone();
        if c.chars().count() > self.max_chars_per_chunk {
            c = format!("{}...", c.chars().take(self.max_chars_per_chunk - 3).collect::<String>());
        }
        let prefix = format!("[{}]", msg.role);
        self.chunks.push(format!("{} {}", prefix, c));
    }

    fn render(&self, max_chars: Option<usize>) -> String {
        let body = self.chunks.join("\n--\n");
        if let Some(max_chars) = max_chars {
            if body.chars().count() > max_chars {
                if max_chars < 3 {
                    return ".".repeat(max_chars);
                }
                return format!("{}...", body.chars().take(max_chars - 3).collect::<String>());
            }
        }
        body
    }

    fn len(&self) -> usize {
        // Longueur cumulee des chunks seules, separateurs exclus.
        self.chunks.iter().map(|c| c.chars().count()).sum()
    }
}

// Booleens a la Python : True / False.
fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

// Demonstration : on couple RollingWindow et ProgressiveSummary.
let mut rw2 = RollingWindow::new(3).unwrap();
let mut sm = ProgressiveSummary::new();
for i in 0..7u32 {
    let m = Message::new(
        if i % 2 == 0 { "user" } else { "assistant" },
        &format!("Le projet X en est a l'iteration {} et les tests passent presque tous sauf un edge case.", i),
    );
    for evicted in rw2.append(m) {
        sm.ajouter(&evicted);
    }
}
println!("Fenetre finale :");
println!("{}", rw2.render());
println!();
println!("Resume cumulatif ({} caracteres) :", sm.len());
println!("{}", sm.render(Some(300)));
println!("Limite de rendu 2 caracteres respecte le plafond : {}.", py_bool(sm.render(Some(2)).chars().count() == 2));

Fenetre finale :


[user] Le projet X en est a l'iteration 4 et les tests passent presque tous sauf un edge case.


[assistant] Le projet X en est a l'iteration 5 et les tests passent presque tous sauf un edge case.


[user] Le projet X en est a l'iteration 6 et les tests passent presque tous sauf un edge case.


Resume cumulatif (358 caracteres) :


[user] Le projet X en est a l'iteration 0 et les tests passent presque tous sauf un ...


... (3 outputs left out)

[user] Le projet X en est a l'iteration 2 et les tests passent presque tous sauf un ...


--


[assistant] Le proj...


Limite de rendu 2 caracteres respecte le plafond : True.


**Lecture de la sortie.** Sur 7 messages avec une fenetre de 3, **4 messages ont ete evicques** et verses dans le resume cumulatif. Le resume preserve l'information ancienne sous forme tronquee (80 chars par chunk) ; le lecteur voit que les **3 messages les plus recents** sont dans la fenetre et que les 4 precedents survivent sous forme compressee. Le résumé est une troncature et une concaténation, pas un résumé sémantique. Les chunks internes restent stockés : seul le texte rendu est plafonné.

## 4. Politique d'eviction -- FIFO, LRU, importance-scored

La fenetre glissante est FIFO par construction. Mais on peut choisir une **politique d'eviction** : quand la memoire est pleine, **lequel** sacrifier ?

- **FIFO** : le plus ancien (comportement de la fenetre glissante ci-dessus).
- **LRU** (Least Recently Used) : celui qu'on n'a pas lu/cite depuis le plus longtemps. Necessite un compteur d'acces par message.
- **Importance-scored** : celui dont l'importance est la plus basse. Necessite un score (calcule par le LLM, par l'usage, par les outils appeles, etc.).

Implementation : `LRUMemory` et `ImportanceMemory` comme variantes de la fenetre glissante (memes operations, politique de remplacement differente). Cote Rust, la LRU travaille par identite d'instance (`Rc<Message>` + comparaison de pointeurs), comme la cle `id(msg)` du jumeau Python.

In [5]:
use std::rc::Rc;

#[derive(Clone, Debug)]
struct Message {
    role: String,   // user / assistant / system / tool
    content: String,
    ts: f64,
    importance: f64,
}

impl Message {
    fn next_ts() -> f64 {
        // Horloge monotone de substitution au time.time() Python (l'ordre seul compte).
        static NEXT_TS: std::sync::atomic::AtomicU64 = std::sync::atomic::AtomicU64::new(0);
        NEXT_TS.fetch_add(1, std::sync::atomic::Ordering::SeqCst) as f64
    }

    fn new(role: &str, content: &str) -> Self {
        Message { role: role.to_string(), content: content.to_string(), ts: Self::next_ts(), importance: 1.0 }
    }

    fn new_with(role: &str, content: &str, ts: f64, importance: f64) -> Self {
        Message { role: role.to_string(), content: content.to_string(), ts, importance }
    }
}

// LRU PAR IDENTITE : equivalent de l'OrderedDict Python cle id(msg).
// En Rust, l'identite d'instance = comparaison de pointeurs sur Rc<Message>.
struct LRUMemory {
    max: usize,
    store: Vec<Rc<Message>>,
    #[allow(dead_code)]
    evicted: Vec<Rc<Message>>,
}

impl LRUMemory {
    fn new(max_messages: usize) -> Result<Self, String> {
        if max_messages < 1 {
            return Err("max_messages >= 1 requis.".to_string());
        }
        Ok(LRUMemory { max: max_messages, store: Vec::new(), evicted: Vec::new() })
    }

    fn append(&mut self, msg: Rc<Message>) -> Vec<Rc<Message>> {
        // Reinsertion de la meme instance : ordre inchange, touch() la promeut.
        if self.store.iter().any(|m| Rc::ptr_eq(m, &msg)) {
            return Vec::new();
        }
        let mut ev = Vec::new();
        if self.store.len() >= self.max {
            let head = self.store.remove(0);
            self.evicted.push(head.clone());
            ev.push(head);
        }
        self.store.push(msg);
        ev
    }

    fn touch(&mut self, msg: &Rc<Message>) {
        if let Some(pos) = self.store.iter().position(|m| Rc::ptr_eq(m, msg)) {
            let m = self.store.remove(pos);
            self.store.push(m);  // move_to_end
        }
    }

    fn ids(&self) -> Vec<usize> {
        self.store.iter().map(|m| Rc::as_ptr(m) as *const () as usize).collect()
    }

    fn len(&self) -> usize {
        self.store.len()
    }

    fn render(&self) -> String {
        self.store.iter()
            .map(|m| format!("[{}] {}", m.role, m.content))
            .collect::<Vec<_>>()
            .join("\n")
    }
}

struct ImportanceMemory {
    max: usize,
    items: Vec<Message>,
    #[allow(dead_code)]
    evicted: Vec<Message>,
}

impl ImportanceMemory {
    fn new(max_messages: usize) -> Result<Self, String> {
        if max_messages < 1 {
            return Err("max_messages >= 1 requis.".to_string());
        }
        Ok(ImportanceMemory { max: max_messages, items: Vec::new(), evicted: Vec::new() })
    }

    fn append(&mut self, msg: Message) -> Vec<Message> {
        let mut ev = Vec::new();
        if self.items.len() >= self.max {
            // min sur le couple (importance, ts) : le PREMIER minimum gagne,
            // comme le min Python sur key=lambda m: (m.importance, m.ts).
            let victim_idx = self.items.iter().enumerate()
                .min_by(|a, b| {
                    a.1.importance.partial_cmp(&b.1.importance).unwrap()
                        .then(a.1.ts.partial_cmp(&b.1.ts).unwrap())
                })
                .map(|(i, _)| i)
                .unwrap();
            let victim = self.items.remove(victim_idx);
            self.evicted.push(victim.clone());
            ev.push(victim);
        }
        self.items.push(msg);
        ev
    }

    fn len(&self) -> usize {
        self.items.len()
    }

    fn render(&self) -> String {
        self.items.iter()
            .map(|m| format!("[{}] (imp={:.2}) {}", m.role, m.importance, m.content))
            .collect::<Vec<_>>()
            .join("\n")
    }
}

// Booleens a la Python : True / False.
fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

// Demonstration : 5 messages dans une LRU de taille 3.
let mut lru = LRUMemory::new(3).unwrap();
let mut msgs: Vec<Rc<Message>> = Vec::new();
for i in 0..5u32 {
    let m = Rc::new(Message::new("user", &format!("message #{}", i)));
    msgs.push(m.clone());
    lru.append(m);
}
println!("Apres 5 insertions, contenu LRU (du plus ancien au plus recent) :");
println!("{}", lru.render());
println!();
println!("On touche msgs[1], deja evince -> sans effet : touch() ne ressuscite pas un message sorti.");
lru.touch(&msgs[1]);
println!("{}", lru.render());
println!();
println!("On touche msgs[2], le LRU courant (encore en memoire) -> il devient MRU.");
lru.touch(&msgs[2]);
println!("{}", lru.render());
println!();
let ev = lru.append(Rc::new(Message::new("user", "message #5")));
println!("Insertion de #5 -> le message evince est : {}", ev[0].content);
println!("Contenu final (le message touche a survecu a l'eviction) :");
println!("{}", lru.render());

// Demonstration importance-scored : un message important survit a un message recu mais banal.
let mut imp = ImportanceMemory::new(3).unwrap();
for (contenu, importance) in [("a", 1.0), ("b", 1.0), ("c", 1.0), ("d", 1.0),
                              ("PENSEE-CLE : ne jamais committer un secret", 10.0)] {
    let m = Message { role: "user".to_string(), content: contenu.to_string(),
                      ts: Message::next_ts(), importance };
    imp.append(m);
}
println!();
println!("ImportanceMemory apres 5 insertions :");
println!("{}", imp.render());

// Controle d'identite : une reinsertion de la meme instance ne cree pas de doublon.
let mut identity_probe = LRUMemory::new(2).unwrap();
let first = Rc::new(Message::new_with("user", "meme valeur", 10.0, 1.0));
identity_probe.append(first.clone());
let no_duplicate_eviction = identity_probe.append(first.clone());
let equal_but_distinct = Rc::new(Message::new_with("user", "meme valeur", 10.0, 1.0));
identity_probe.append(equal_but_distinct.clone());
let third = Rc::new(Message::new_with("user", "troisieme", 11.0, 1.0));
let identity_victim = identity_probe.append(third.clone());
identity_probe.touch(&equal_but_distinct);
let order_after_touch = identity_probe.ids();
identity_probe.touch(&first);  // absent : ne doit pas modifier l'ordre
// Le libelle du jumeau Python compare la liste rendue par append a l'instance :
// c'est toujours False par construction. Cote Rust, on imprime le CALCUL honnete
// du meme libelle : l'instance de meme valeur est-elle celle qui a ete evincee ?
assert!(identity_victim.len() == 1 && Rc::ptr_eq(&identity_victim[0], &first));
let b_reinsert = identity_probe.len() == 2 && no_duplicate_eviction.is_empty();
let b_valeur_evincee = identity_victim.iter().any(|v| Rc::ptr_eq(v, &equal_but_distinct));
let b_promut = order_after_touch == vec![Rc::as_ptr(&third) as *const () as usize,
                                         Rc::as_ptr(&equal_but_distinct) as *const () as usize];
let b_absent = identity_probe.ids() == order_after_touch;
println!("Controle LRU par identite : reinsertion meme instance sans eviction={}; instance de meme valeur evincee={}; touch present promeut={}; touch absent sans effet={}.",
         py_bool(b_reinsert), py_bool(b_valeur_evincee), py_bool(b_promut), py_bool(b_absent));

Apres 5 insertions, contenu LRU (du plus ancien au plus recent) :


[user] message #2


[user] message #3


[user] message #4


On touche msgs[1], deja evince -> sans effet : touch() ne ressuscite pas un message sorti.


[user] message #2


... (12 outputs left out)

[user] (imp=1.00) c


[user] (imp=1.00) d


[user] (imp=10.00) PENSEE-CLE : ne jamais committer un secret


Controle LRU par identite : reinsertion meme instance sans eviction=True; instance de meme valeur evincee=False; touch present promeut=True; touch absent sans effet=True.


**Lecture de la sortie.** Le contrôle d’identité montre qu’une réinsertion ne crée pas de doublon ; un objet distinct de mêmes champs reste distinct, puis l’éviction et Touch sont vérifiés par identité. Sur la mémoire LRU : apres 5 insertions dans une fenetre de 3, le contenu est `#2 / #3 / #4` (du plus ancien au plus recent). On touche d'abord `msgs[1]`, **deja evince** : `touch()` ne fait rien, la memoire reste `#2 / #3 / #4` -- un message sorti ne se ressuscite pas. On touche ensuite `msgs[2]`, le **LRU courant** : il passe en queue (`#3 / #4 / #2`). L'insertion de `#5` evince alors `#3` (le LRU) et non `#2` : **le message touche a survecu a l'eviction**, la memoire finale est `#4 / #2 / #5`. La promotion MRU est donc visible dans la sortie, et non seulement annoncee. Sur l'`ImportanceMemory` : apres 5 insertions, le message `PENSEE-CLE` (importance 10) survit alors qu'il a ete insere en dernier, parce que les 4 precedents (tous importance 1) ont ete preferes a tour de role ; a la 5e insertion, le LRU-parmi-les-faibles est elimine et le message important reste.

## 5. Memory paging (style Generative Agents / MemGPT)

Le **memory paging** est le mecanisme des systemes type MemGPT (Packer et al., 2023) et Generative Agents (Park et al., 2023) : la memoire est decoupee en **pages** (sous-unites adressables), et l'agent ne voit a chaque tour que la **page courante** dans son contexte. Pour acceder a une page non-courante, il **appelle un outil** de retrieval qui swap-in la page demandee.

L'analogie est volontairement systeme d'exploitation : la memoire de l'agent est un espace d'adressage virtuel, le contexte du LLM est la memoire physique (RAM), et le swap est un outil de retrieval.

Implementation simplifiee ici : `PagedMemory` tient une liste de pages ; `current_page` est celle injectee dans le prompt. `swap_to(idx)` charge une autre page (cout symbolique simule).

In [6]:
#[derive(Clone, Debug)]
#[allow(dead_code)]
struct Message {
    role: String,   // user / assistant / system / tool
    content: String,
    ts: f64,
    importance: f64,
}

impl Message {
    fn next_ts() -> f64 {
        // Horloge monotone de substitution au time.time() Python (l'ordre seul compte).
        static NEXT_TS: std::sync::atomic::AtomicU64 = std::sync::atomic::AtomicU64::new(0);
        NEXT_TS.fetch_add(1, std::sync::atomic::Ordering::SeqCst) as f64
    }

    fn new(role: &str, content: &str) -> Self {
        Message { role: role.to_string(), content: content.to_string(), ts: Self::next_ts(), importance: 1.0 }
    }
}

#[derive(Clone, Debug)]
struct Page {
    title: String,
    messages: Vec<Message>,
}

impl Page {
    fn render(&self) -> String {
        let body = self.messages.iter()
            .map(|m| format!("  [{}] {}", m.role, m.content))
            .collect::<Vec<_>>()
            .join("\n");
        format!("=== Page: {} ({} msg) ===\n{}", self.title, self.messages.len(), body)
    }
}

struct PagedMemory {
    pages: Vec<Page>,
    current: usize,
}

impl PagedMemory {
    fn new() -> Self {
        PagedMemory { pages: Vec::new(), current: 0 }
    }

    fn add_page(&mut self, page: Page) -> usize {
        self.pages.push(page);
        self.pages.len() - 1
    }

    fn swap_to(&mut self, idx: i64) -> Result<&Page, String> {
        if !(idx >= 0 && (idx as usize) < self.pages.len()) {
            return Err(format!("page index {} hors limites (0..{}).", idx, self.pages.len() - 1));
        }
        self.current = idx as usize;
        Ok(&self.pages[self.current])
    }

    fn current_page(&self) -> &Page {
        &self.pages[self.current]
    }

    fn total_messages(&self) -> usize {
        self.pages.iter().map(|p| p.messages.len()).sum()
    }
}

// Demonstration : 3 pages sur 3 sujets, swap entre elles.
let mut pm = PagedMemory::new();
pm.add_page(Page {
    title: "onboarding".to_string(),
    messages: vec![
        Message::new("user", "Bonjour, je m'appelle Alice."),
        Message::new("assistant", "Bonjour Alice, bienvenue."),
    ],
});
pm.add_page(Page {
    title: "bug-investigation".to_string(),
    messages: vec![
        Message::new("user", "Le test #42 echoue avec une AssertionError."),
        Message::new("assistant", "Voyons la trace... la ligne 17."),
        Message::new("user", "C'etait bien la ligne 17, merci."),
    ],
});
pm.add_page(Page {
    title: "preferences".to_string(),
    messages: vec![
        Message::new("user", "Je prefere qu'on reponde en francais."),
        Message::new("assistant", "Compris, je repondrai en francais."),
    ],
});

println!("Memoire paginee : {} pages, {} messages au total.", pm.pages.len(), pm.total_messages());
println!("Page courante (idx={}) : {}", pm.current, pm.current_page().title);
println!();
for (idx, page) in pm.pages.iter().enumerate() {
    let marker = if idx == pm.current { " <-- courante" } else { "" };
    println!("Page #{}: {}{}", idx, page.title, marker);
}
println!();
println!("Swap vers la page preferences (idx=2)...");
match pm.swap_to(2) {
    Ok(p) => println!("{}", p.render()),
    Err(e) => println!("{}", e),
};

Memoire paginee : 3 pages, 7 messages au total.


Page courante (idx=0) : onboarding


Page #0: onboarding <-- courante


Page #1: bug-investigation


Page #2: preferences


Swap vers la page preferences (idx=2)...


=== Page: preferences (2 msg) ===


  [user] Je prefere qu'on reponde en francais.


  [assistant] Compris, je repondrai en francais.


**Lecture de la sortie.** La memoire contient 3 pages (7 messages au total). La page courante est `onboarding` (idx 0). Apres `swap_to(2)`, la page `preferences` est chargee dans le contexte. **Le gain** : un agent qui n'a besoin que du contexte `preferences` n'envoie au LLM que les 2 messages de cette page (au lieu des 7), mais les 3 pages restent en mémoire ; le calcul de coût ultérieur suppose le swap gratuit.

## 6. Mesure -- unites de cout modelisees en caracteres

Comparons des couts synthetiques exprimes en caracteres (proxy pedagogique, pas jetons mesures) entre une memoire **non bornee** (liste classique) et une memoire **paginee** + resume progressif, sur deux conversations synthetiques (100 et 1000 tours).

In [7]:
use std::collections::VecDeque;

// Conversation synthetique : messages alternes user/assistant.
const MSG_AVG_LEN: u64 = 120;  // caracteres par message (proxy de tokens ~1:4)
const WINDOW: u64 = 10;        // taille de la fenetre glissante
const RESUME_MAX: u64 = 200;   // budget du resume cumulatif (chars)
const PAGE_SIZE: u64 = 5;      // taille d'une page

#[derive(Clone, Debug)]
#[allow(dead_code)]
struct Message {
    role: String,
    content: String,
    ts: f64,
    importance: f64,
}

impl Message {
    fn next_ts() -> f64 {
        // Horloge monotone de substitution au time.time() Python (l'ordre seul compte).
        static NEXT_TS: std::sync::atomic::AtomicU64 = std::sync::atomic::AtomicU64::new(0);
        NEXT_TS.fetch_add(1, std::sync::atomic::Ordering::SeqCst) as f64
    }

    fn new(role: &str, content: &str) -> Self {
        Message { role: role.to_string(), content: content.to_string(), ts: Self::next_ts(), importance: 1.0 }
    }
}

struct RollingWindow {
    max: usize,
    buf: VecDeque<Message>,
    evicted_total: usize,
}

impl RollingWindow {
    fn new(max_messages: usize) -> Result<Self, String> {
        if max_messages < 1 {
            return Err("max_messages >= 1 requis.".to_string());
        }
        Ok(RollingWindow { max: max_messages, buf: VecDeque::new(), evicted_total: 0 })
    }

    fn append(&mut self, msg: Message) -> Vec<Message> {
        let mut ev = Vec::new();
        if self.buf.len() == self.max {
            ev.push(self.buf.pop_front().unwrap());
        }
        self.buf.push_back(msg);
        self.evicted_total += ev.len();
        ev
    }

    fn len(&self) -> usize {
        self.buf.len()
    }

    fn render(&self) -> String {
        self.buf.iter()
            .map(|m| format!("[{}] {}", m.role, m.content))
            .collect::<Vec<_>>()
            .join("\n")
    }
}

struct ProgressiveSummary {
    chunks: Vec<String>,
    max_chars_per_chunk: usize,
}

impl ProgressiveSummary {
    fn with_max(max_chars_per_chunk: usize) -> Self {
        ProgressiveSummary { chunks: Vec::new(), max_chars_per_chunk }
    }

    fn ajouter(&mut self, msg: &Message) {
        // HOOK production : remplacer par summarize(msg.content) (appel LLM).
        let mut c = msg.content.clone();
        if c.chars().count() > self.max_chars_per_chunk {
            c = format!("{}...", c.chars().take(self.max_chars_per_chunk - 3).collect::<String>());
        }
        let prefix = format!("[{}]", msg.role);
        self.chunks.push(format!("{} {}", prefix, c));
    }

    fn render(&self, max_chars: Option<usize>) -> String {
        let body = self.chunks.join("\n--\n");
        if let Some(max_chars) = max_chars {
            if body.chars().count() > max_chars {
                if max_chars < 3 {
                    return ".".repeat(max_chars);
                }
                return format!("{}...", body.chars().take(max_chars - 3).collect::<String>());
            }
        }
        body
    }

    fn len(&self) -> usize {
        // Longueur cumulee des chunks seules, separateurs exclus.
        self.chunks.iter().map(|c| c.chars().count()).sum()
    }
}

#[derive(Clone, Debug)]
struct Page {
    title: String,
    messages: Vec<Message>,
}

struct PagedMemory {
    pages: Vec<Page>,
    current: usize,
}

impl PagedMemory {
    fn new() -> Self {
        PagedMemory { pages: Vec::new(), current: 0 }
    }

    fn add_page(&mut self, page: Page) -> usize {
        self.pages.push(page);
        self.pages.len() - 1
    }

    #[allow(dead_code)]
    fn swap_to(&mut self, idx: i64) -> Result<&Page, String> {
        if !(idx >= 0 && (idx as usize) < self.pages.len()) {
            return Err(format!("page index {} hors limites (0..{}).", idx, self.pages.len() - 1));
        }
        self.current = idx as usize;
        Ok(&self.pages[self.current])
    }

    #[allow(dead_code)]
    fn current_page(&self) -> &Page {
        &self.pages[self.current]
    }

    #[allow(dead_code)]
    fn total_messages(&self) -> usize {
        self.pages.iter().map(|p| p.messages.len()).sum()
    }
}

// Separateur de milliers : virgule tous les trois chiffres depuis la droite.
fn group(n: u64) -> String {
    let s = n.to_string();
    let bytes = s.as_bytes();
    let len = bytes.len();
    let mut out = String::new();
    for (i, b) in bytes.iter().enumerate() {
        if i > 0 && (len - i) % 3 == 0 {
            out.push(',');
        }
        out.push(*b as char);
    }
    out
}

// Cout total (chars) des trois memoires sur n_turns messages.
fn gains_pour(n_turns: u64) -> (u64, u64, u64) {
    let conv: Vec<Message> = (0..n_turns)
        .map(|i| {
            Message::new(
                if i % 2 == 0 { "user" } else { "assistant" },
                &format!("Message #{:03} - {}", i, "x".repeat(MSG_AVG_LEN as usize)),
            )
        })
        .collect();

    // Memoire non bornee : on injecte TOUT l'historique a chaque tour.
    let total_unbounded: u64 = (0..n_turns).map(|i| MSG_AVG_LEN * (i + 1)).sum();

    // Fenetre glissante + resume cumulatif (compact).
    let mut rw = RollingWindow::new(WINDOW as usize).unwrap();
    let mut sm = ProgressiveSummary::with_max(40);
    let mut total_windowed: u64 = 0;
    for m in conv.iter() {
        for evicted in rw.append(m.clone()) {
            sm.ajouter(&evicted);
        }
        total_windowed += WINDOW * MSG_AVG_LEN + RESUME_MAX.min(sm.len() as u64);
    }

    // Memory paging a pages de PAGE_SIZE, swap automatique selon le sujet simule.
    let mut pm = PagedMemory::new();
    let mut i = 0u64;
    while i < n_turns {
        let end = (i + PAGE_SIZE).min(n_turns);
        pm.add_page(Page {
            title: format!("page-{}", i / PAGE_SIZE),
            messages: conv[i as usize..end as usize].to_vec(),
        });
        i += PAGE_SIZE;
    }
    let total_paged = n_turns * (PAGE_SIZE * MSG_AVG_LEN + RESUME_MAX);

    (total_unbounded, total_windowed, total_paged)
}

const PROFILS: [u64; 2] = [100, 1000];
let resultats: Vec<(u64, u64, u64, u64)> = PROFILS.iter()
    .map(|&n| { let (a, b, c) = gains_pour(n); (n, a, b, c) })
    .collect();

println!("{}", "=".repeat(84));
println!("Conversation synthetique : {} chars/message, fenetre W={}, pages P={}", MSG_AVG_LEN, WINDOW, PAGE_SIZE);
println!("{}", "=".repeat(84));
for &(n, ub, w, p) in &resultats {
    println!("{:>5} tours : non bornee = {:>11} | fenetre = {:>10} (x{:>5.1}) | paging = {:>10} (x{:>5.1})",
             n, group(ub), group(w), ub as f64 / w as f64, group(p), ub as f64 / p as f64);
}
println!();
println!("Verdict honnete (G.2) : le cout UNI-side varie avec le profil de la conversation.");
println!("  - non bornee : O(N) par tour -> explosion quadratique sur la duree.");
println!("  - fenetre : O(W) par tour -> lineaire borne, mais perte d'info ancienne.");
println!("  - paging : O(P) par tour + cout de swap, lineaire borne, info preservee par pagination.");
let gw: Vec<f64> = resultats.iter().map(|&(_, ub, w, _)| ub as f64 / w as f64).collect();
let gp: Vec<f64> = resultats.iter().map(|&(_, ub, _, p)| ub as f64 / p as f64).collect();
println!("  - le gain croit avec N : fenetre x{:.1} -> x{:.1}, paging x{:.1} -> x{:.1}, a W et P fixes.", gw[0], gw[1], gp[0], gp[1]);

Conversation synthetique : 120 chars/message, fenetre W=10, pages P=5


  100 tours : non bornee =     606,000 | fenetre =    137,690 (x  4.4) | paging =     80,000 (x  7.6)


 1000 tours : non bornee =  60,060,000 | fenetre =  1,397,690 (x 43.0) | paging =    800,000 (x 75.1)


Verdict honnete (G.2) : le cout UNI-side varie avec le profil de la conversation.


  - non bornee : O(N) par tour -> explosion quadratique sur la duree.


  - fenetre : O(W) par tour -> lineaire borne, mais perte d'info ancienne.


  - paging : O(P) par tour + cout de swap, lineaire borne, info preservee par pagination.


  - le gain croit avec N : fenetre x4.4 -> x43.0, paging x7.6 -> x75.1, a W et P fixes.


**Lecture des mesures.** Aux profils de 100 et 1000 tours, la mémoire non bornée totalise respectivement 606 000 et 60 060 000 unités, contre 137 690 et 1 397 690 pour fenêtre + résumé, puis 80 000 et 800 000 pour paging. Les facteurs affichés sont 4,4/43,0 et 7,6/75,1. Ce sont des résultats du calcul synthétique à paramètres fixés, pas des jetons réellement envoyés : la fenêtre est facturée pleine dès le premier tour, les préfixes/rôles sont omis et le swap est gratuit. Les chunks de résumé et archives restent stockés en O(N).


## 7. Trois exercices

Les exercices suivent la convention C.1 : pas d'erreur volontaire, le notebook tourne de bout en bout memes exercices non completes. Chaque exercice introduit un mecanisme non couvert dans la demonstration ci-dessus.

<a id="exercice-1"></a>

### Exercice 1 -- fenetre parametrique et cout marginal

Implemente une fonction `cout_marginal(window_size, n_turns, msg_len)` qui calcule le cout total d'une fenetre glissante sur `n_turns` messages, avec une taille de fenetre `window_size` et une longueur moyenne `msg_len`. Convention précise : la fenêtre est payée pleine dès le premier tour, donc coûtFenêtre=N×W×L ; coûtNonBorné=L×N(N+1)/2. Le seuil strict coûtFenêtre < coûtNonBorné/2 équivaut à W<(N+1)/4. Le minimum admissible vaut 1 quand il en existe un ; calcule aussi la plus grande fenêtre admissible et traite N=1, où aucune fenêtre entière ne satisfait le seuil.

**Indice** : dans le modèle retenu, le coût de la fenêtre réserve dès le premier tour `window_size * msg_len`, même pendant le remplissage. Ce choix de budget constant est une enveloppe, pas la longueur réelle de la fenêtre à ce stade. Le cout par tour d'une memoire non bornee est lineaire : `i * msg_len` au tour i. Trace les deux courbes et trouve leur intersection.

In [8]:
// Exercice 1 : cout total + cout marginal d'une fenetre glissante.
// Renvoie (cout_fenetre, seuil_atteint, w_seuil).
// seuil_atteint = 1 si cout marginal < seuil, 0 sinon.
// w_seuil = taille de fenetre minimale qui satisfait le critere.
fn cout_marginal(window_size: i64, n_turns: i64, msg_len: i64) -> (i64, i64, i64) {
    // TODO etudiant : implementer le calcul.
    let _ = (window_size, n_turns, msg_len);
    (0, 0, 0)
}

let (_cout, _seuil, _w) = cout_marginal(10, 100, 120);
println!("Exercice 1 - cout marginal : cout_total={}, seuil_atteint={}, w_seuil={}  (a completer)", _cout, _seuil, _w);

Exercice 1 - cout marginal : cout_total=0, seuil_atteint=0, w_seuil=0  (a completer)


<a id="exercice-2"></a>

### Exercice 2 -- eviction LRU sur la memoire vectorielle de NB-14

Reprends `MemoireVectorielle` du notebook 14 (`MyIA.AI.Notebooks/GenAI/Texte/14_Persistent_Memory.ipynb`) et ajoute-lui une politique d'eviction **importance + recence** : a chaque `ajouter`, si `len(entrees) > max_entrees`, evince l'entree de **score pondéré combinant récence et importance**. La recence est mesuree par l'age en secondes. L'importance est la **longueur de la lecon** (proxy : une lecon detaillee est plus importante qu'un acquittement sec, comme demontre dans NB-14).

**Indice** : le `score` peut etre `0.5 * (1 / (1 + age_en_secondes)) + 0.5 * (len(lecon) / max_lecon_len)`. A chaque ajout, recalcule les scores et evince le plus bas. Verifie que la lecon correcte (`PENSEE-CLE : ne jamais committer un secret`) reste en memoire apres plusieurs insertions.

In [9]:
// Exercice 2 : memoire vectorielle paginee, inspiree de NB-14 (BoWEmbedder.cosine).
#[allow(dead_code)]
struct MemoireVectoriellePaginee {
    embedder: Box<dyn Fn(&str) -> Vec<f64>>,        // rend un vecteur pour une specification
    max_entrees: usize,                             // defaut 50
    entrees: Vec<(String, String, f64, Vec<f64>)>,  // (spec, lecon, ts, vecteur)
}

#[allow(dead_code)]
impl MemoireVectoriellePaginee {
    fn new(embedder: Box<dyn Fn(&str) -> Vec<f64>>) -> Self {
        MemoireVectoriellePaginee { embedder, max_entrees: 50, entrees: Vec::new() }
    }

    fn ajouter(&mut self, spec: &str, lecon: &str, ts: Option<f64>) {
        // TODO etudiant : ajouter une entree avec ts, vecteur, et evincer si > max_entrees.
        let _ = (spec, lecon, ts);
    }

    fn score(&self, entree: &(String, String, f64, Vec<f64>), max_lecon_len: usize, now: f64) -> f64 {
        // TODO etudiant : combiner recence et importance en un score [0, 1].
        let _ = (entree, max_lecon_len, now);
        0.0
    }

    fn retroceder(&self, spec_query: &str, k: usize) -> Vec<(String, String)> {
        // TODO etudiant : reprendre le code de NB-14 (BoWEmbedder.cosine).
        let _ = (spec_query, k);
        Vec::new()
    }
}

println!("Exercice 2 - MemoireVectoriellePaginee : a completer.");

Exercice 2 - MemoireVectoriellePaginee : a completer.


<a id="exercice-3"></a>

### Exercice 3 (avance) -- transfer cross-paging multi-domaines

Etends `PagedMemory` pour gerer des **domaines** (par exemple : `code`, `math`, `preferences`). Chaque page porte un `domaine`, et la methode `swap_to_domain(dom)` charge la **derniere page vue** de ce domaine (au lieu de demander un index numerique). Teste : un utilisateur parle de code pendant 10 tours, puis de preferences pendant 5 tours, puis revient au code -- le swap-to-domain re-charge-t-il bien la derniere page code vue ?

**Indice** : ajoute un champ `domaine` a `Page`, et une table `last_seen: HashMap<String, usize>` (`dom -> page_idx`) dans `PagedMemory`. La methode `swap_to_domain(dom)` consulte `last_seen[dom]` et appelle `swap_to(...)`.

In [11]:
use std::collections::HashMap;

#[derive(Clone, Debug)]
#[allow(dead_code)]
struct Message {
    role: String,
    content: String,
    ts: f64,
    importance: f64,
}

#[allow(dead_code)]
impl Message {
    fn next_ts() -> f64 {
        // Horloge monotone de substitution au time.time() Python (l'ordre seul compte).
        static NEXT_TS: std::sync::atomic::AtomicU64 = std::sync::atomic::AtomicU64::new(0);
        NEXT_TS.fetch_add(1, std::sync::atomic::Ordering::SeqCst) as f64
    }

    fn new(role: &str, content: &str) -> Self {
        Message { role: role.to_string(), content: content.to_string(), ts: Self::next_ts(), importance: 1.0 }
    }
}

#[derive(Clone, Debug)]
#[allow(dead_code)]
struct Page {
    title: String,
    messages: Vec<Message>,
    // TODO etudiant : ajouter un champ domaine par page.
}

#[allow(dead_code)]
struct PagedMemoryMultiDomaine {
    pages: Vec<Page>,
    last_seen: HashMap<String, usize>,  // domaine -> index de la derniere page vue
    current: usize,
}

#[allow(dead_code)]
impl PagedMemoryMultiDomaine {
    fn new() -> Self {
        PagedMemoryMultiDomaine { pages: Vec::new(), last_seen: HashMap::new(), current: 0 }
    }

    fn add_page(&mut self, page: Page, domaine: &str) -> usize {
        // TODO etudiant : ajouter la page, mettre a jour last_seen[domaine] = idx.
        let _ = (page, domaine);
        0
    }

    fn swap_to_domain(&mut self, domaine: &str) -> Option<&Page> {
        // TODO etudiant : swap vers last_seen[domaine] si present, sinon erreur explicite.
        let _ = domaine;
        None
    }
}

println!("Exercice 3 - PagedMemoryMultiDomaine : a completer.");

Exercice 3 - PagedMemoryMultiDomaine : a completer.


## 8. À retenir et suite

On a introduit les **trois mecanismes** de la memoire d'agent bornee (fenetre glissante, resume tronque, eviction LRU/importance) et le **memory paging** style Generative Agents / MemGPT. Les couts calcules sur deux profils (100 puis 1000 tours) montre qu'une fenêtre de 10 réduit le coût modélisé en caractères par 4.4 puis 43.0, et un paging par pages de 5 réduit ce coût modélisé par 7.6 puis 75.1, par rapport a une memoire non bornee : le facteur croît avec N sous les conventions fixées. Ce ne sont pas des tokens réellement envoyés ou facturés : W×120 est compté dès le premier tour, P×120+200 à chaque tour, préfixes/rôles sont omis et le swap est gratuit. Fenêtre affichée bornée ne signifie pas stockage borné : chunks et archives croissent en O(N).

**Limites honnetes (G.2) :**

- **Resume simplifie** : la demonstration utilise une troncature a 80 chars par message evince. Un vrai systeme appelle un LLM (`summarize(msg) -> str`) -- cf le `# HOOK` dans `ProgressiveSummary.ajouter`. Le slot est marque, le code de production reste a brancher.
- **Cout de swap au paging** : `swap_to(idx)` est suppose gratuit dans la demo (pas de cout de retrieval). En production (MemGPT), le swap implique un round-trip LLM pour decider quelle page charger -- c'est le cout cache du paging, qui peut annuler le gain si la conversation switch souvent de sujet.
- **Importance subjective** : la politique importance-scored utilise un proxy (`longueur de la lecon`) qui n'est qu'un proxy. En production, l'importance est scoree par le LLM ou par des signaux d'usage (messages les plus retrocedes, reponses les mieux notees par l'utilisateur).
- **Jumeaux** : voir [le carnet Python d'origine](14b_Paginated_Memory-Python.ipynb) et [le jumeau C# BCL](14b_Paginated_Memory-CSharp.ipynb), qui implémentent les mêmes démonstrations sans API externe.

**Suite de l'epic #18220 (plis 2-3) :**

- **#18211** -- exploiter un LLM en production (routage, cache, cout, deploiement progressif) -- la memoire paginee est l'un des leviers du deploiement progressif (garder en cache les pages les plus accedees).
- **#18213** -- protocoles d'agents A2A et securite MCP -- la pagination devient un vecteur d'attaque (un agent malveillant pourrait demander le swap d'une page contenant des secrets).
- **Suite logique 14c** -- Generative Agents complet : reflexion, importance scoree par LLM, planification journaliere. Cf Park et al., 2023.

**References :** Park et al., 2023 (*Generative Agents: Interactive Simulacra of Human Behavior*) ; Packer et al., 2023 (*MemGPT: Towards LLMs as Operating Systems*) ; NB-12 (Reflexion), NB-13 (routeur agentique), NB-14 (memoire vectorielle persistente).

> **Provenance** : dossier [#18220](https://github.com/jsboige/CoursIA/issues/18220) sur les lacunes pédagogiques. La ligne « mémoire paginée d’agent » était absente du tableau Agents après vérification directe : `07-KernelMemory-*` couvre l’ETL documentaire, `10b/10f/10_LocalLlama/22` mentionnent PagedAttention côté cache KV d’inférence, et rien dans `ML/DataScienceWithAgents/` ne traite de fenêtre glissante, résumé progressif ou éviction de messages anciens.


### Récapitulatif des trois exercices

1. [Coût marginal et fenêtres admissibles](#exercice-1) : garder le stub non résolu et calculer la borne inférieure et supérieure selon la convention écrite.
2. [Mémoire vectorielle](#exercice-2) : combiner récence et longueur de leçon par score pondéré, avec horloge et embedding déterministes.
3. [Pagination par domaine](#exercice-3) : retrouver la dernière page effectivement vue.

## Vérifiez votre compréhension

<details><summary>Un résumé par troncature comprend-il le sens du message ?</summary>

Non : le texte est coupé à une longueur fixée puis concaténé, sans reformulation sémantique.

</details>

<details><summary>Le paging borne-t-il la mémoire totale ?</summary>

Non. Il borne ce qui est présenté comme contexte à un tour ; les pages archivées et les chunks du résumé restent conservés en O(N).

</details>

<details><summary>Les profils 100 et 1000 mesurent-ils une facture de tokens ?</summary>

Non. Ils appliquent des unités de coût modélisées en caractères sous les conventions documentées, sans tokenizer ni service de génération.

</details>

## Pour aller plus loin

- [Carnet Python d'origine](14b_Paginated_Memory-Python.ipynb)
- [Jumeau C# BCL](14b_Paginated_Memory-CSharp.ipynb)
- [Notebook 14 — mémoire persistante](14_Persistent_Memory.ipynb)
- [Notebook 12 — réflexivité](12_Test_Time_Scaling.ipynb) et [13 — agents/outils](13_Agentic_Orchestration.ipynb)
- Park et al. (2023), *Generative Agents* ; Packer et al. (2023), *MemGPT*.
